# Лабораторная работа 6. Решающие деревья и композиции алгоритмов

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 5 |
| Опора на лекции | лекция 5: решающее дерево (опр. 5.1), критерии информативности (опр. 5.2–5.4), прирост информации (опр. 5.5) и его неотрицательность (утв. 5.6), стрижка по цене сложности (опр. 5.8), бутстреп и бэггинг (опр. 5.10–5.11), разброс усреднённого предсказания (утв. 5.12), AdaBoost и вес слабого классификатора (теорема 5.15); лекция 4: смещение–разброс, скользящий контроль, SRM |
| Трудоёмкость | 2 ч аудиторно (части 1–4) + 6 ч самостоятельно |

## Цель работы

Реализовать решающее дерево с нуля, включая выбор расщепления по приросту информации; проверить утверждение 5.6 численно; понять, почему бэггинг снижает разброс, а бустинг — смещение, и увидеть это в разложении из работы 5; реализовать AdaBoost по теореме 5.15 и сравнить поведение случайного леса и градиентного бустинга при росте числа деревьев; научиться корректно измерять важность признаков.

## Что нужно сдать

Заполненный ноутбук `lab06_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.model_selection import cross_val_score, train_test_split, KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=6)
describe_variant(variant)

---
# Часть 1. Критерии информативности

Определения 5.2–5.4: для множества объектов $R$ с долями классов $p_k$

$$
H(R) = -\sum_k p_k \ln p_k, \qquad
\mathrm{Gini}(R) = 1 - \sum_k p_k^2, \qquad
D(R) = \frac1{|R|}\sum_{i \in R}(y_i - \bar y_R)^2 .
$$

Прирост информации (опр. 5.5) при разбиении $R = R_L \sqcup R_R$:

$$
\mathrm{Gain} = \Phi(R) - \frac{|R_L|}{|R|}\Phi(R_L) - \frac{|R_R|}{|R|}\Phi(R_R) .
$$

Утверждение 5.6: для энтропии и Джини $\mathrm{Gain} \ge 0$ **всегда** —
это следствие строгой вогнутости обеих функций. Проверим численно.

In [ ]:
def entropy(y):
    """H(R) = -sum p_k ln p_k, соглашение 0 ln 0 = 0."""
    raise NotImplementedError


def gini(y):
    """Gini(R) = 1 - sum p_k^2."""
    raise NotImplementedError


def variance(y):
    """D(R) -- критерий дисперсии для регрессии."""
    raise NotImplementedError


def gain(y, mask, phi):
    """Прирост информации при разбиении множества по булевой маске."""
    raise NotImplementedError


# TODO: 1) постройте графики энтропии, Джини и доли ошибок min(p, 1-p)
#          для двух классов как функций p_1;
#       2) проверьте утверждение 5.6 численно: на 20000 случайных разбиениях
#          случайных множеств найдите для энтропии, Джини и доли ошибок
#          минимальный Gain, долю отрицательных Gain и долю НУЛЕВЫХ Gain.
#          У какого критерия нулевой прирост встречается заметно чаще?
print("критерий вашего варианта:", variant["criterion"])

> **Вывод.** Для каких критериев прирост оказался неотрицательным всегда, а для какого — нет? Почему доля ошибок ведёт себя иначе и что это значит для построения дерева?
>
> *(ваш ответ здесь)*

---
# Часть 2. Решающее дерево с нуля

Алгоритм (§«Построение дерева» лекции 5): в вершине перебираются все признаки
и все пороги, выбирается расщепление с максимальным приростом; рекурсия
останавливается по глубине, по числу объектов или при нулевом приросте.

Реализуйте дерево для задачи из вашего варианта (`variant["own_tree_task"]`)
с критерием `variant["criterion"]`. Порогами достаточно брать середины между
соседними уникальными значениями признака.

In [ ]:
class Node:
    """Узел дерева: либо лист со значением, либо предикат [x_feature <= threshold]."""
    # TODO


class MyDecisionTree:
    """Решающее дерево: классификация (Джини/энтропия) или регрессия (дисперсия).

    fit(X, y): рекурсивно растит дерево.
    _best_split(X, y): перебирает признаки и пороги (середины между соседними
        различными значениями), возвращает (gain, feature, threshold).
    Остановка: max_depth, min_samples_split, min_samples_leaf, нулевой прирост,
        все объекты одного класса.
    predict(X), n_leaves().
    """
    # TODO


print("задача вашего варианта:", variant["own_tree_task"],
      "| критерий:", variant["criterion"])

### Задание 2.2. Сверка со `sklearn` и граница решения

Обучите своё дерево и `sklearn` на одних данных при одинаковых ограничениях и
сравните: качество, число листьев, границу решения при разных глубинах.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# TODO: 1) подготовьте данные под задачу своего варианта (make_moons для
#          классификации либо своя гладкая функция + шум для регрессии);
#       2) для max_depth in [1, 2, 3, 5, 8, 12, None] сравните своё дерево
#          и sklearn: качество на контроле и число листьев;
#       3) для классификации нарисуйте границы решения при глубинах 1, 3, 6, None.

> **Вывод.** Совпало ли качество вашего дерева со `sklearn`? Как выглядит граница решения и почему она такая? Начиная с какой глубины начинается переобучение?
>
> *(ваш ответ здесь)*

---
# Часть 3. Стрижка

Определение 5.8: $Q_\alpha(T) = Q(T, X^\ell) + \alpha\,|\mathrm{leaves}(T)|$.
Как отмечено в замечании к нему, это ровно принцип SRM (лекция 4) для семейства
поддеревьев: $\alpha$ играет роль штрафа за сложность и подбирается скользящим
контролем.

В `sklearn` это параметр `ccp_alpha`, а метод `cost_complexity_pruning_path`
выдаёт последовательность «эффективных» $\alpha$, при которых дерево меняется.
Сравните стрижку с ограничением глубины (`variant["pruning"]` указывает,
какой способ разбирать подробно).

In [ ]:
print("способ регуляризации вашего варианта:", variant["pruning"])

# TODO: 1) вырастите полное дерево и получите cost_complexity_pruning_path;
#       2) для сетки alpha посчитайте число листьев, оценку скользящего контроля
#          и качество на отложенной выборке;
#       3) выберите alpha* по скользящему контролю, постройте два графика
#          (листья от alpha; качество от alpha) и сравните дерево до и после стрижки.

---
# Часть 4. Бэггинг: почему усреднение помогает

Утверждение 5.12: если $a_1,\dots,a_B$ одинаково распределены, имеют дисперсию
$\sigma^2$ и попарную корреляцию $\rho$, то

$$
\mathrm{Var}\Bigl(\frac1B\sum_b a_b(x)\Bigr) \;=\; \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2 .
$$

Отсюда сразу видно: увеличением $B$ можно убрать только второе слагаемое, а
первое, $\rho\sigma^2$, — предел, ниже которого усреднение не опускается.
Случайный лес атакует именно $\rho$: случайный подбор признаков при расщеплении
делает деревья менее похожими друг на друга.

Проверьте формулу численно и совместите её с разложением смещение–разброс
из работы 5.

In [ ]:
from sklearn.ensemble import (AdaBoostClassifier, AdaBoostRegressor, BaggingClassifier,
                              BaggingRegressor, ExtraTreesClassifier, ExtraTreesRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)

D = 5
F_TRUE5 = lambda X: np.sin(2 * X[:, 0]) + 0.7 * X[:, 1] ** 2 - 0.5 * X[:, 2] * X[:, 3]
SIGMA = 0.35


def sample5(n, g):
    X = g.uniform(-2, 2, size=(n, D))
    return X, F_TRUE5(X) + g.normal(0, SIGMA, n)


# TODO: (1) методикой из работы 5 (часть 2) посчитайте смещение^2 и разброс для
#           трёх моделей: одиночное дерево, бэггинг (max_features=1.0),
#           случайный лес (max_features=0.4). Данные ПЯТИМЕРНЫЕ -- иначе
#           ограничение числа признаков не имеет смысла.
#       (2) проверьте утверждение 5.12 в фиксированной точке x0:
#           N_TREES_REP раз генерируйте СВОЮ обучающую выборку (случайность
#           выборки -- это и есть источник корреляции между деревьями!),
#           внутри каждой обучите B_MAX деревьев на бутстреп-подвыборках;
#           оцените sigma^2 и rho, сравните измеренный Var(среднего по первым B)
#           с формулой rho*sigma^2 + (1-rho)/B * sigma^2.

> **Вывод.** Насколько формула утверждения 5.12 согласуется с экспериментом? Что происходит с разбросом и смещением при переходе от одного дерева к лесу? Зачем случайный лес ограничивает число признаков при расщеплении, если это ухудшает каждое отдельное дерево?
>
> *(ваш ответ здесь)*

---
# Часть 5. AdaBoost по теореме 5.15

Теорема 5.15: при весах $w_i = \exp(-y_i F_{t-1}(x_i))$ и взвешенной ошибке
$\mathrm{err}_t$ оптимальный вес слабого классификатора равен

$$
\alpha_t = \frac12\ln\frac{1 - \mathrm{err}_t}{\mathrm{err}_t},
$$

после чего веса обновляются как
$w_i \leftarrow w_i\exp(-\alpha_t y_i b_t(x_i))$ и нормируются.

Реализуйте AdaBoost с пнями (деревьями глубины 1) и сравните со `sklearn`.
Дополнительно проверьте, что формула для $\alpha_t$ действительно доставляет
минимум — численной минимизацией по $\alpha$.

In [ ]:
def adaboost_fit(X, y, n_estimators=200, seed=0):
    """AdaBoost для y из {-1, +1}. На каждом шаге:
       1) обучить пень с весами w (DecisionTreeClassifier(max_depth=1), sample_weight=w);
       2) посчитать взвешенную ошибку err_t;
       3) alpha_t = 0.5 * ln((1 - err_t)/err_t)   -- теорема 5.15;
       4) w_i <- w_i exp(-alpha_t y_i b_t(x_i)), нормировать.
       Возвращает (список пней, массив alpha, историю по шагам).
    """
    raise NotImplementedError


def adaboost_predict(X, stumps, alphas):
    raise NotImplementedError


# TODO: 1) обучите свой AdaBoost на make_moons и сравните с AdaBoostClassifier;
#       2) проверьте формулу теоремы 5.15: сравните alpha по формуле с численной
#          минимизацией sum_i w_i exp(-alpha y_i b(x_i)) по alpha;
#       3) постройте графики: ошибка композиции и экспоненциальная потеря от T,
#          а также err_t и alpha_t по шагам.

> **Вывод.** Совпал ли $\alpha_t$ из формулы с численным минимумом? Как ведёт себя $\mathrm{err}_t$ с ростом $t$ и почему? Почему обучающая ошибка композиции падает до нуля, хотя каждый пень ошибается почти в 40 % случаев?
>
> *(ваш ответ здесь)*

---
# Часть 6. Лес против бустинга: что происходит при росте числа деревьев

Принципиальное различие: бэггинг усредняет **независимо обученные** алгоритмы и
потому от числа деревьев не переобучается (утв. 5.12: разброс монотонно убывает
к $\rho\sigma^2$); бустинг строит **зависимую** последовательность, каждый шаг
уменьшает смещение, и после некоторого $T$ начинается переобучение.

Сравните ансамбли из вашего варианта (`variant["ensembles"]`) по числу деревьев.

In [ ]:
# TODO: для ансамблей из variant["ensembles"] постройте зависимость качества
#       на обучении и на контроле от числа деревьев (сетка [1,2,5,10,25,50,100,200,400]).
#       Сведите в таблицу лучшее качество, оптимальное число деревьев и качество при 400.
print("ансамбли вашего варианта:", ", ".join(variant["ensembles"]))

> **Вывод.** Какие из ваших ансамблей выходят на плато, а какие переобучаются с ростом числа деревьев? Как это связано с тем, что усредняется — независимые или зависимые алгоритмы?
>
> *(ваш ответ здесь)*

---
# Часть 7. Важность признаков

Два способа (`variant["importance"]` указывает вашу пару):

* **по приросту (impurity-based)**, `feature_importances_`: суммарный
  взвешенный прирост информации по всем расщеплениям, использующим признак;
* **перестановочная (permutation)**: качество на контроле после случайной
  перестановки значений признака — насколько оно упало;
* **drop-column**: модель переобучается без признака.

Первый способ вычисляется бесплатно, но **смещён**: он завышает важность
признаков с большим числом различных значений, потому что у них больше
кандидатов в пороги. Проверьте это в чистом эксперименте: добавим к данным
случайный шумовой признак с большим числом уникальных значений и случайный
бинарный — оба заведомо бесполезны.

In [ ]:
from sklearn.inspection import permutation_importance

# TODO: 1) постройте выборку из 5 признаков: два полезных (вещественный и бинарный)
#          и три ЗАВЕДОМО бесполезных (вещественный шум, бинарный шум, 5 категорий);
#          цель зависит только от двух первых;
#       2) обучите RandomForest и посчитайте важности тремя способами:
#          feature_importances_, permutation_importance, drop-column;
#       3) сведите в таблицу, постройте горизонтальную столбчатую диаграмму
#          и сравните суммарную важность шумовых признаков по каждому способу.

> **Вывод.** Какую важность получили заведомо бесполезные признаки при каждом способе? Почему шумовой вещественный признак «важнее» шумового бинарного? Какой способ вы будете использовать и почему?
>
> *(ваш ответ здесь)*

---
# Часть 8. Своя выборка

Сравните на индивидуальной выборке: одиночное дерево (со стрижкой, подобранной
скользящим контролем) и все ансамбли вашего варианта с подобранными
гиперпараметрами. Оцените важность признаков лучшей модели и объясните результат
содержательно — с точки зрения предметной области вашего датасета.

In [ ]:
from sklearn.model_selection import GridSearchCV

data = load_personal(variant)
# TODO: 1) подберите скользящим контролем гиперпараметры одиночного дерева
#          (max_depth, min_samples_leaf) и всех ансамблей вашего варианта;
#       2) сведите в таблицу лучшие параметры, оценку CV и качество на контроле;
#       3) для лучшей модели постройте перестановочную важность 10 главных
#          признаков и объясните результат содержательно.

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Почему дерево не может провести диагональную границу, и что с этим делают на практике? Приведите два способа.
2. В бэггинге деревья намеренно не стригут, а в бустинге берут пни глубины 1–3. Объясните оба решения через разложение смещение–разброс.
3. Из утверждения 5.12 следует, что $\mathrm{Var}(\bar a) \to \rho\sigma^2$ при $B \to \infty$. Какими двумя способами случайный лес уменьшает $\rho$ и чем за это платит?
4. Ваш случайный лес показал важность 0.22 у признака «номер договора». Что вы проверите в первую очередь?
5. Почему у случайного леса можно ставить `n_estimators=1000` не задумываясь, а у градиентного бустинга — нельзя?

### Домашнее задание

1. **Своя стрижка по цене сложности.** Реализуйте для своего дерева алгоритм слабейшего звена (weakest link pruning): для каждого внутреннего узла $v$ вычислите $g(v) = \bigl(Q(v) - Q(T_v)\bigr)/\bigl(|\mathrm{leaves}(T_v)| - 1\bigr)$, срежьте узел с минимальным $g$, повторяйте до корня. Полученная последовательность $\alpha$ и поддеревьев должна совпасть с `cost_complexity_pruning_path` из `sklearn`. Проверьте совпадение и объясните, почему поддеревья, оптимальные для разных $\alpha$, образуют вложенную цепочку.

2. **Градиентный бустинг для регрессии с нуля.** Реализуйте бустинг над квадратичной потерей: $F_0 = \bar y$, затем на каждом шаге обучайте дерево на **остатках** $r_i = y_i - F_{t-1}(x_i)$ и обновляйте $F_t = F_{t-1} + \eta\,h_t$. Сравните с `GradientBoostingRegressor` при тех же `learning_rate`, `max_depth`, `n_estimators`. Постройте зависимость качества на контроле от $\eta$ и $T$ и покажите эмпирически, что произведение $\eta \cdot T$ приблизительно постоянно на линии оптимума — объясните, почему малый шаг требует больше итераций.